<center> <h2> Using Kafkanator Fairness metrics table  </h2> </center>

<p> In this notebook you will learn to use kafkanator create_dataviz method to quickly get a summary of fairness metrics in your predictive model.
For this, we will use the ADULT dataset, a well known dataset in fairness domain to predict income based in some attributes, including sensistive ones such as gender and race. </p>
<h3> Outline : </h3>
<ol>
    <li> Data Downloading </li>
    <li> Data Preprocessing </li>
    <li> Training Model </li>
    <li> Executing the fairness metrics table method .</li>
    <ol>
        <li>A single sensitive attribute . </li>
        <li>Multiple sensitive attributes. </li>
</ol>

In [1]:
%pip install ucimlrepo


[notice] A new release of pip is available: 23.2.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


<h3> 1. Data Downloading </h3>

In [1]:
from ucimlrepo import fetch_ucirepo 
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from kafkanator import *
from collections import Counter
import xgboost as xgb
import pandas as pd
import numpy as np
from kafkanator.fairness import *
import kafkanator.util as ut

In [2]:
# fetch dataset 
adult = fetch_ucirepo(id=2) 
# data (as pandas dataframes) 
data = adult.data.features 
y = adult.data.targets 
# metadata 
print(adult.metadata) 
# variable information 
print(adult.variables) 
# Data cleaning
y['income'] = y['income'].apply(lambda x: x.replace('.',''))
data['sex'] = data['sex'].str.strip()
data['race'] = data['race'].str.strip()
data['marital-status'] = data['marital-status'].str.strip()
data['native-country'] = data['native-country'].str.strip()

{'uci_id': 2, 'name': 'Adult', 'repository_url': 'https://archive.ics.uci.edu/dataset/2/adult', 'data_url': 'https://archive.ics.uci.edu/static/public/2/data.csv', 'abstract': 'Predict whether annual income of an individual exceeds $50K/yr based on census data. Also known as "Census Income" dataset. ', 'area': 'Social Science', 'tasks': ['Classification'], 'characteristics': ['Multivariate'], 'num_instances': 48842, 'num_features': 14, 'feature_types': ['Categorical', 'Integer'], 'demographics': ['Age', 'Income', 'Education Level', 'Other', 'Race', 'Sex'], 'target_col': ['income'], 'index_col': None, 'has_missing_values': 'yes', 'missing_values_symbol': 'NaN', 'year_of_dataset_creation': 1996, 'last_updated': 'Tue Sep 24 2024', 'dataset_doi': '10.24432/C5XW20', 'creators': ['Barry Becker', 'Ronny Kohavi'], 'intro_paper': None, 'additional_info': {'summary': "Extraction was done by Barry Becker from the 1994 Census database.  A set of reasonably clean records was extracted using the fol

In [3]:
data

,age,workclass,fnlwgt,education,education-num,marital-status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,native-country
0,39,State-gov,77516,Bachelors,13,Never-married,Adm-clerical,Not-in-family,White,Male,2174,0,40,United-States
1,50,Self-emp-not-inc,83311,Bachelors,13,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,0,13,United-States
2,38,Private,215646,HS-grad,9,Divorced,Handlers-cleaners,Not-in-family,White,Male,0,0,40,United-States
3,53,Private,234721,11th,7,Married-civ-spouse,Handlers-cleaners,Husband,Black,Male,0,0,40,United-States
4,28,Private,338409,Bachelors,13,Married-civ-spouse,Prof-specialty,Wife,Black,Female,0,0,40,Cuba
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
48837,39,Private,215419,Bachelors,13,Divorced,Prof-specialty,Not-in-family,White,Female,0,0,36,United-States
48838,64,NaN,321403,HS-grad,9,Widowed,NaN,Other-relative,Black,Male,0,0,40,United-States
48839,38,Private,374983,Bachelors,13,Married-civ-spouse,Prof-specialty,Husband,White,Male,0,0,50,United-States
48840,44,Private,83891,Bachelors,13,Divorced,Adm-clerical,Own-child,Asian-Pac-Islander,Male,5455,0,40,United-States


<h3> 2. Data preprocessing </h3>

In [4]:
data['race'] = data['race'].apply(lambda x : 1 if x=='White' else 0 )

In [5]:
data[data['sex']=='Male'].shape

(32650, 14)

In [6]:
data[data['sex']=='Female'].shape

(16192, 14)

In [7]:
categorical_columns = ['workclass', 'education',  'occupation', 'relationship', 'race', 'sex','marital-status','native-country']

# Create an instance of LabelEncoder
label_encoder = LabelEncoder()

# Encode each categorical column and create new feature columns
for col in categorical_columns:
    encoded_col_name = f"{col}_encoded"
    data[encoded_col_name] = label_encoder.fit_transform(data[col])

In [8]:
Y = label_encoder.fit_transform(y['income'])

In [9]:
X = data[['age','workclass_encoded', 'education_encoded','education-num','marital-status_encoded','occupation_encoded',	'relationship_encoded','race_encoded','sex_encoded','capital-loss','capital-gain','hours-per-week','native-country_encoded']]

<h2> 3. Training the model </h2>

In [10]:
X_train, X_test, y_train, y_test = train_test_split( X, Y, test_size=0.3)

In [11]:
#classifier = RandomForestClassifier(n_estimators=100, random_state=42)
classifier = xgb.XGBClassifier(objective="binary:logistic")
classifier.fit(X_train, y_train)
y_pred = classifier.predict(X_test)

In [12]:
y_pred

array([0, 0, 0, ..., 0, 0, 0])

In [13]:
sens_metrics = X_test['sex_encoded']

In [14]:
X_test

,age,workclass_encoded,education_encoded,education-num,marital-status_encoded,occupation_encoded,relationship_encoded,race_encoded,sex_encoded,capital-loss,capital-gain,hours-per-week,native-country_encoded
15802,44,2,15,10,0,4,1,1,1,0,0,50,39
46007,37,4,11,9,2,8,2,0,0,0,0,35,39
47259,44,4,15,10,0,14,4,1,1,0,0,35,9
36563,51,4,11,9,0,8,1,1,1,0,0,80,39
20837,58,2,9,13,2,4,0,1,1,0,0,40,39
...,...,...,...,...,...,...,...,...,...,...,...,...,...
34633,33,4,9,13,0,4,4,1,0,0,0,40,39
23358,37,4,9,13,2,3,0,1,1,0,0,40,39
8807,49,2,9,13,4,10,1,1,0,0,0,50,39
18261,34,4,11,9,4,3,3,1,1,0,0,40,39


<h2> 4. Executing the fairness metrics table method </h2>
<p> This method receives as parameter a numpy matrix with sensitive attribute, prediction and reality columns. You can specify
one sensitive attribute in an array, or multiple. As indicated in followint subsections </p>

In [15]:
fairness_analysis = np.transpose(np.array([X_test['sex_encoded'].values, y_pred,y_test]))
fairness_analysis_df = pd.DataFrame(data=fairness_analysis,columns=['sex_encoded','prediction','reality'])

<h2> 4.1. One sensitive attribute : Gender </h2>

In [16]:
sp = fairness_metrics_table(fairness_analysis_df,['sex_encoded'],'prediction','reality')

ks  {'0': 0.07933541017653167, '1': 0.26804228501727995}
 lcosl  ['0', '1', 'DELTA']
sta party output  {(0,): 0.07933541017653167, (1,): 0.26804228501727995}
 df shape  (5, 3)


In [17]:
sp.style.apply(ut.row_highlighting,axis=1)

 ma is  0.11582910967652271


,0,1,DELTA
DEMOGRAPHIC PARITY - P1,0.079335,0.268042,0.188707
EQUAL OPPORTUNITY - TPR,0.570058,0.685887,0.115829
PREDICTIVE PARITY - PPV,0.777487,0.775882,0.001605
"EQUALIZED ODDS - (TPR,FPR)","0.019795062878435025,0.5700575815738963","0.0862144420131291,0.685886691250419","0.06641937913469408,0.11582910967652271"
DISPARATE IMPACT - PREVALENCE,0.108204,0.303212,0.356858


<h2> You can also obtain normal fairness metrics by using kafkanator fairness module methods: </h2>

In [23]:
eo = equal_opportunity(concatenated,'sex_encoded','prediction','reality')
eod = equalized_odds(concatenated,'sex_encoded','prediction','reality')
pp = predictive_parity(concatenated,'sex_encoded','prediction','reality')

In [24]:
eo

{0: 0.6095617529880478, 1: 0.6773636991028296}

In [25]:
eod

{0: '0.023304107060452238,0.6095617529880478',
 1: '0.08151466974996387,0.6773636991028296'}

In [26]:
pp

{0: 0.7518427518427518, 1: 0.776810447170558}

<h3> 4.2 Multiple sensitive attributes : gender and race </h3>
<p> Just build a numpy matrix containing sensitive attributes, reality and predictions columns, then pass an array containning
an arbitrary set of sensitive attributes </p>

In [20]:
fairness_analysis = np.transpose(np.array([X_test['sex_encoded'].values,X_test['race_encoded'].values, y_pred,y_test]))
concatenated = pd.DataFrame(data=fairness_analysis,columns=['sex_encoded','race_encoded','prediction','reality'])
sp = fairness_metrics_table(concatenated,['sex_encoded','race_encoded'],'prediction','reality')

ks  {'0,0': 0.0429769392033543, '0,1': 0.08831908831908832, '1,0': 0.18734388009991673, '1,1': 0.2792636332059743}
 lcosl  ['0,0', '0,1', '1,0', '1,1', 'DELTA']
sta party output  {(0, 0): 0.0429769392033543, (0, 1): 0.08831908831908832, (1, 0): 0.18734388009991673, (1, 1): 0.2792636332059743}
 df shape  (5, 5)


In [21]:
sp

,"0,0","0,1","1,0","1,1",DELTA
DEMOGRAPHIC PARITY - P1,0.042977,0.088319,0.187344,0.279264,0.236287
EQUAL OPPORTUNITY - TPR,0.410959,0.595982,0.645283,0.689845,0.278887
PREDICTIVE PARITY - PPV,0.731707,0.782991,0.76,0.777363,0.051284
"EQUALIZED ODDS - (TPR,FPR)","0.012485811577752554,0.410958904109589","0.021681804863756225,0.5959821428571429","0.009195993286003672,0.1850232387475539","0.09072478459199189,0.6898454746136865",NaN
DISPARATE IMPACT - PREVALENCE,0.07652,0.116032,0.659472,0.314693,NaN
